In [ ]:
from models import GraspAnalysisResults
from utils import GraspAnalysisUtils
from pathlib import Path
import numpy as np


In [ ]:
# Find folder containing CSV files for analysis
data_folder = GraspAnalysisUtils.select_folder()


In [ ]:
# Check if there as csv files in the chosen directory
dir_path = Path(data_folder)
if any(file.suffix.lower() == ".csv" for file in dir_path.iterdir() if file.is_file()):
    files = [str(file) for file in dir_path.glob("*.csv")]
    print("CSV files located")
else:
    raise RuntimeError("No CSV files found in chosen folder")

In [ ]:
# Plot Dialog Selection
chosen_plots = GraspAnalysisUtils.select_desired_plots()
print(f"User wants to create: {chosen_plots}")

In [ ]:
# Assignment of variables
SAMPLE_SIZE = 200
OFFSET = 2000
MIN_FORCE_THRESHOLD = 10

In [ ]:
# Performing Analysis

for file_path in files:
    
    filename = Path(file_path).name

    force_data = GraspAnalysisUtils.load_and_preprocess_data(file_path)

    grasps = GraspAnalysisUtils.detect_grasp_regions(force_data, SAMPLE_SIZE, MIN_FORCE_THRESHOLD)

    number_of_grasps = len(grasps)

    rolling_avg = []
    rolling_std = []
    rolling_median = []

    if number_of_grasps == 0:
        Warning(f"No grasps detected in file: {filename}")
        continue

    avg_forces = []
    for grasp in grasps:
        grasp = GraspAnalysisUtils.calculate_grasp_force(force_data[grasp.start_idx:grasp.end_idx], grasp, OFFSET)

        avg_forces.append(grasp.avg_force)
        rolling_avg.append(np.mean(avg_forces))
        rolling_std.append(np.std(avg_forces))
        rolling_median.append(np.median(avg_forces))

    # Calculate Statistics 
    mean, std, sem, ts, ci, pi, cv_percent = GraspAnalysisUtils.calculate_grasp_statistics(number_of_grasps, grasps)

    required_samples = GraspAnalysisUtils.calculate_required_samples(mean, std, number_of_grasps)
    
    if required_samples is not None:
        moe, n_required, n_required_buffer, std_dev_interval_lwr, std_dev_interval_upr, n_required_std = required_samples
        
    result = GraspAnalysisResults(
        filename,
        number_of_grasps,
        mean,
        std,
        sem,
        ts,
        ci,
        pi,
        cv_percent,
        rolling_avg,
        rolling_std,
        rolling_median,
        n_required,
        n_required_buffer,
        moe,
        std_dev_interval_lwr,
        std_dev_interval_upr,
        n_required_std
    )

    GraspAnalysisUtils.report_results(result)

    GraspAnalysisUtils.create_grasp_plots(chosen_plots, force_data, result, grasps)